# Predicting the Electronic Band Gap of Inorganic Crystalline Materials from Composition and Structure
### Stage 4 — Data Inspection and Exploratory Analysis

Uses the raw dataset saved by the Stage 3 notebook (`mp_bandgap_raw.csv`, 154,377 records). This notebook does:

1. Formal data inspection (dimensions, dtypes, missing values, duplicates)
2. The deferred stability-filter decision from Stage 3 (`energy_above_hull`), made from the actual distribution
3. A small set of scientifically motivated EDA plots
4. A documented, audited "cleaned" dataset for Stage 5

**No feature engineering, modeling, or performance claims happen here.**


## Load the Stage 3 output

If this is a fresh Colab runtime (session restarted since Stage 3), `mp_bandgap_raw.csv` won't exist in local storage — you'll need to either re-run the Stage 3 notebook first, or mount Google Drive and save/load from there instead. The cell below fails loudly rather than silently generating placeholder data if the file is missing.


In [ ]:
import os
import json
import ast
import shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DRIVE_DIR = "/content/drive/MyDrive/materials_bandgap_project"
RAW_CSV_NAME = "mp_bandgap_raw.csv"
METADATA_NAME = "mp_bandgap_acquisition_metadata.json"

RAW_CSV_PATH = RAW_CSV_NAME
METADATA_PATH = METADATA_NAME

if not os.path.exists(RAW_CSV_PATH):
    # Not in local Colab storage (new/reset runtime) -- try Google Drive instead.
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except ImportError:
        pass  # not running in Colab

    drive_raw_path = os.path.join(DRIVE_DIR, RAW_CSV_NAME)
    drive_meta_path = os.path.join(DRIVE_DIR, METADATA_NAME)

    if os.path.exists(drive_raw_path):
        shutil.copy(drive_raw_path, RAW_CSV_PATH)
        if os.path.exists(drive_meta_path):
            shutil.copy(drive_meta_path, METADATA_PATH)
        print(f"Loaded {RAW_CSV_NAME} from Google Drive ({DRIVE_DIR}).")
    else:
        raise FileNotFoundError(
            f"{RAW_CSV_NAME} not found locally or at {drive_raw_path} on Drive. "
            "Either run the Stage 3 notebook's acquisition cells in this session first, "
            "or save its output to that Drive folder before running Stage 4."
        )

raw_df = pd.read_csv(RAW_CSV_PATH)

# The 'elements' column was saved as a stringified Python list (e.g. "['Fe', 'O']").
# Convert it back to an actual list so later filtering/boolean checks work correctly.
def parse_elements(val):
    if pd.isna(val):
        return None
    try:
        return ast.literal_eval(val)
    except (ValueError, SyntaxError):
        return None

raw_df["elements"] = raw_df["elements"].apply(parse_elements)

if os.path.exists(METADATA_PATH):
    with open(METADATA_PATH) as f:
        acquisition_metadata = json.load(f)
    print("Loaded Stage 3 acquisition metadata:")
    print(f"  Retrieved:      {acquisition_metadata['retrieval_datetime_utc']}")
    print(f"  mp-api version: {acquisition_metadata['mp_api_version']}")
    print(f"  Query params:   {acquisition_metadata['query_parameters']}")
else:
    print("WARNING: acquisition metadata file not found -- provenance for this dataset is incomplete.")

print(f"\nLoaded {raw_df.shape[0]} rows, {raw_df.shape[1]} columns.")


## 1. Data Inspection — Dimensions, Types, Structure


In [ ]:
print("Shape:", raw_df.shape)
print()
raw_df.info()


In [ ]:
raw_df.head()


## 2. Duplicates and Identifier Integrity

`material_id` should be a unique key — if it isn't, something is wrong with the pull. Repeated *formulas* (`formula_pretty`), on the other hand, are expected and scientifically meaningful: they represent distinct polymorphs of the same composition (different structures, different `material_id`s, often different band gaps). These are not duplicates to remove — they're exactly the data points needed to test whether structural descriptors add predictive value over composition alone (Stage 2, secondary RQ1). We just need to be aware of this clustering for the grouped train/test split.


In [ ]:
n_unique_ids = raw_df["material_id"].nunique()
n_rows = raw_df.shape[0]
print(f"Rows: {n_rows}, unique material_id: {n_unique_ids}")
assert n_unique_ids == n_rows, "material_id is not unique — investigate before proceeding."

formula_counts = raw_df["formula_pretty"].value_counts()
n_formulas_with_multiple_polymorphs = (formula_counts > 1).sum()
print(f"Unique formulas: {formula_counts.shape[0]}")
print(f"Formulas with more than one polymorph in this dataset: {n_formulas_with_multiple_polymorphs}")
print("\nMost-duplicated formulas (top 10):")
print(formula_counts.head(10))


## 3. Missing Values

Cross-checking against the Stage 3 quality report to confirm nothing changed between the raw export and this load.


In [ ]:
missing = raw_df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("Columns with missing values:")
print(missing)
print(f"\nTotal rows with any missing value: {raw_df.isna().any(axis=1).sum()}")


## 4. Summary Statistics (numeric columns)


In [ ]:
numeric_cols = ["band_gap", "nsites", "volume", "density", "density_atomic",
                "formation_energy_per_atom", "energy_above_hull", "nelements"]
raw_df[numeric_cols].describe().T


## 5. Target Distribution — `band_gap`

The Stage 3 report already showed this is bimodal-ish: 72,456 metallic (`band_gap == 0`) vs. 81,917 with a positive gap, with a heavily right-skewed positive tail (median 0.081 eV, mean 1.06 eV, max 17.9 eV). Two views: the full distribution (showing the zero-spike), and the positive-only subset zoomed in (showing the actual shape of the semiconductor/insulator gap distribution).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

axes[0].hist(raw_df["band_gap"].dropna(), bins=100, color="#4C72B0")
axes[0].set_title("Band gap — full distribution")
axes[0].set_xlabel("Band gap (eV)")
axes[0].set_ylabel("Count")

positive_gap = raw_df.loc[raw_df["band_gap"] > 0, "band_gap"]
axes[1].hist(positive_gap, bins=100, color="#DD8452")
axes[1].set_title(f"Band gap — non-metals only (n={len(positive_gap)})")
axes[1].set_xlabel("Band gap (eV)")

plt.tight_layout()
plt.show()

print(f"Metallic (band_gap == 0): {(raw_df['band_gap'] == 0).sum()} "
      f"({(raw_df['band_gap'] == 0).mean():.1%})")
print(f"Non-metallic (band_gap > 0): {(raw_df['band_gap'] > 0).sum()} "
      f"({(raw_df['band_gap'] > 0).mean():.1%})")


**Implication for later stages (not acted on yet):** the strong right-skew and zero-inflation mean a plain regression on raw `band_gap` will be dominated by the large low-gap cluster; this is worth revisiting at the modeling stage (Stage 6/7) — e.g. via error metrics reported separately for metals vs. non-metals — rather than transforming the target now with no model in hand to justify it.


## 6. Stability (`energy_above_hull`) — the Deferred Filtering Decision

**Context:** `energy_above_hull` measures thermodynamic metastability (0 = ground state; larger = further from the convex hull, i.e. less thermodynamically favorable). It is commonly used in the literature as a rough proxy for synthesizability, but the commonly cited thresholds vary a great deal by chemistry: some sources put a "typical" cutoff around 70 meV/atom for a majority of experimentally realized metastable phases, others use ~0.1 eV/atom as a general screening threshold, and some note that certain chemistries (e.g. nitrides, oxides under kinetic trapping) can tolerate 0.2–0.3 eV/atom or more, while others (simple metallic alloys) tolerate much less. There is no single universally correct number.

First, let's see what different candidate cutoffs would actually do to *this* dataset.


In [ ]:
hull_stats = raw_df["energy_above_hull"].describe()
print(hull_stats)
print()

candidate_cutoffs = [0.05, 0.07, 0.1, 0.15, 0.2, 0.3, 0.5, 1.0]
n_total = raw_df["energy_above_hull"].notna().sum()

print(f"{'Cutoff (eV/atom)':>18} | {'Retained':>10} | {'% Retained':>10} | {'% Removed':>10}")
print("-" * 58)
for cutoff in candidate_cutoffs:
    retained = (raw_df["energy_above_hull"] <= cutoff).sum()
    pct_retained = retained / n_total * 100
    print(f"{cutoff:>18} | {retained:>10} | {pct_retained:>9.1f}% | {100 - pct_retained:>9.1f}%")


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(raw_df["energy_above_hull"].clip(upper=1.0), bins=100, color="#55A868")
ax.set_xlabel("energy_above_hull (eV/atom), clipped at 1.0 for readability")
ax.set_ylabel("Count")
ax.set_title("Energy above hull distribution (long tail beyond 1.0 eV/atom not shown)")
for cutoff in [0.1, 0.2, 0.3]:
    ax.axvline(cutoff, color="red", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()


**Decision: no blanket `energy_above_hull` cutoff is applied to the main dataset.** Reasoning:

1. **The research target doesn't require thermodynamic stability.** The task is predicting the *computed* band gap from composition/structure — a well-defined DFT quantity for any relaxed structure, metastable or not. Stability filtering matters for a *synthesizability* study; this isn't one.
2. **The "right" cutoff is chemistry-dependent, and applying one globally would introduce a selection bias correlated with composition** — the literature above notes some chemical families (nitrides, oxides) tolerate more metastability than others (metallic alloys), so a single eV/atom threshold would disproportionately remove certain chemistries rather than applying evenly across the population. That's the same kind of convenience-driven, composition-linked filtering the project's chemical-scope decision (Stage 2) deliberately avoided.
3. Given the retention table above, applying even a middle-of-the-road cutoff (0.1–0.2 eV/atom) would discard a large, non-random fraction of the dataset for a reason unrelated to data quality.

**What *is* applied:** a small, purely data-quality exclusion — rows in the extreme upper tail of `energy_above_hull` (beyond the 99.5th percentile) are flagged and removed, on the grounds that these represent pathological/highly unphysical configurations rather than meaningful "very metastable but potentially real" materials. The exact threshold and count removed are computed and logged below, not assumed.

`energy_above_hull` itself remains in the cleaned dataset as an available covariate for later stages (already flagged in the Stage 2 leakage review as "not leakage, test cautiously").


In [ ]:
hull_upper_cutoff = raw_df["energy_above_hull"].quantile(0.995)
n_extreme_hull = (raw_df["energy_above_hull"] > hull_upper_cutoff).sum()
pct_extreme_hull = n_extreme_hull / n_total * 100

print(f"99.5th percentile of energy_above_hull: {hull_upper_cutoff:.4f} eV/atom")
print(f"Records above this threshold: {n_extreme_hull} ({pct_extreme_hull:.2f}% of records with a valid value)")


## 7. Correlation Among Numeric Fields

Checking for obvious redundancy before Stage 5 feature engineering — e.g. `density` and `density_atomic` are likely to be closely related by construction.


In [ ]:
corr_cols = ["band_gap", "nsites", "volume", "density", "density_atomic",
             "formation_energy_per_atom", "energy_above_hull"]
corr = raw_df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax, vmin=-1, vmax=1)
ax.set_title("Correlation matrix — available numeric fields")
plt.tight_layout()
plt.show()


## 8. Feature–Target Relationships (available fields only)

Composition-derived elemental descriptors (electronegativity, atomic radius, etc.) don't exist yet — that's Stage 5. Here we only look at what's already in the raw summary data: density, crystal symmetry, and transition-metal content.


In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
hb = ax.hexbin(raw_df["density"], raw_df["band_gap"], gridsize=60, cmap="viridis", mincnt=1)
ax.set_xlabel("Density (g/cm3)")
ax.set_ylabel("Band gap (eV)")
ax.set_title("Band gap vs. density (hexbin, n={:,})".format(raw_df.shape[0]))
fig.colorbar(hb, label="Count")
plt.tight_layout()
plt.show()


In [ ]:
D_BLOCK = set([
    "Sc","Ti","V","Cr","Mn","Fe","Co","Ni","Cu","Zn",
    "Y","Zr","Nb","Mo","Tc","Ru","Rh","Pd","Ag","Cd",
    "Hf","Ta","W","Re","Os","Ir","Pt","Au","Hg",
])

def contains_tm(els):
    return bool(set(els) & D_BLOCK) if isinstance(els, list) else False

raw_df["is_tm_containing"] = raw_df["elements"].apply(contains_tm)

fig, ax = plt.subplots(figsize=(6, 4.5))
sns.boxplot(
    data=raw_df, x="is_tm_containing", y="band_gap", ax=ax,
    showfliers=False,
)
ax.set_xticklabels(["Main-group only", "Transition-metal-containing"])
ax.set_ylabel("Band gap (eV)")
ax.set_title("Band gap distribution: TM-containing vs. main-group-only compounds")
plt.tight_layout()
plt.show()

print(raw_df.groupby("is_tm_containing")["band_gap"].describe())


In [ ]:
top_systems = raw_df["crystal_system"].value_counts().head(7).index

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.boxplot(
    data=raw_df[raw_df["crystal_system"].isin(top_systems)],
    x="crystal_system", y="band_gap", ax=ax, showfliers=False,
    order=top_systems,
)
ax.set_ylabel("Band gap (eV)")
ax.set_xlabel("Crystal system (7 most common)")
ax.set_title("Band gap distribution by crystal system")
plt.tight_layout()
plt.show()


**Note on these plots:** these are descriptive, exploratory observations at the data-inspection stage — not model results, and not yet a claim that any of these relationships will hold up as predictive once composition-derived descriptors are added in Stage 5. Any apparent pattern here is a hypothesis to carry into the chemical-interpretation stage, not a conclusion.


## 9. Structural Outlier Check (`nsites`)

Very large unit cells are legitimate but rare, and worth knowing about before feature engineering (e.g. some composition-weighted descriptors are computed per formula unit and shouldn't be affected, but it's worth confirming that's true rather than assuming).


In [ ]:
nsites_99 = raw_df["nsites"].quantile(0.99)
n_large_cells = (raw_df["nsites"] > nsites_99).sum()

print(raw_df["nsites"].describe())
print(f"\n99th percentile of nsites: {nsites_99:.0f}")
print(f"Records above the 99th percentile: {n_large_cells} (flagged for awareness, NOT removed)")

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(raw_df["nsites"].clip(upper=int(nsites_99 * 1.5)), bins=80, color="#8172B2")
ax.set_xlabel("nsites (clipped for readability)")
ax.set_title("Unit cell size distribution")
plt.tight_layout()
plt.show()


## 10. Build the Audited, Cleaned Dataset for Stage 5

Every exclusion applied from raw data to this point, in order, with counts — nothing is dropped without being logged here.


In [ ]:
audit_trail = []
working_df = raw_df.copy()
audit_trail.append({"step": "raw_load", "n_rows": len(working_df)})

# 1. Drop rows with missing band_gap (nothing to predict)
before = len(working_df)
working_df = working_df[working_df["band_gap"].notna()]
audit_trail.append({
    "step": "drop_missing_band_gap",
    "n_removed": before - len(working_df),
    "n_rows_after": len(working_df),
})

# 2. Drop deprecated (expected to remove 0, given the filtered query path used in Stage 3)
before = len(working_df)
working_df = working_df[~working_df["deprecated"]]
audit_trail.append({
    "step": "drop_deprecated",
    "n_removed": before - len(working_df),
    "n_rows_after": len(working_df),
})

# 3. Drop extreme energy_above_hull outliers only (data-quality exclusion, NOT a
#    stability/synthesizability filter — see Section 6 reasoning above)
before = len(working_df)
working_df = working_df[working_df["energy_above_hull"] <= hull_upper_cutoff]
audit_trail.append({
    "step": "drop_extreme_energy_above_hull_outliers",
    "threshold_ev_per_atom": float(hull_upper_cutoff),
    "n_removed": before - len(working_df),
    "n_rows_after": len(working_df),
})

print(json.dumps(audit_trail, indent=2))

CLEANED_CSV_PATH = "mp_bandgap_stage4_cleaned.csv"
working_df.to_csv(CLEANED_CSV_PATH, index=False)

with open("mp_bandgap_stage4_audit_trail.json", "w") as f:
    json.dump(audit_trail, f, indent=2)

print(f"\nSaved cleaned dataset ({working_df.shape[0]} rows) to {CLEANED_CSV_PATH}")


In [ ]:
# Persist Stage 4 outputs to Google Drive so later stages can find them
import shutil, os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = "/content/drive/MyDrive/materials_bandgap_project"
    os.makedirs(SAVE_DIR, exist_ok=True)
    for fname in ["mp_bandgap_stage4_cleaned.csv", "mp_bandgap_stage4_audit_trail.json"]:
        if os.path.exists(fname):
            shutil.copy(fname, os.path.join(SAVE_DIR, fname))
            print(f"Copied {fname} to Drive")
        else:
            print(f"WARNING: {fname} not found in this session")
except ImportError:
    print("Not running in Colab -- files remain in the local working directory.")


### Stage 4 ends here.

No feature engineering, no train/test split, no modeling. The next stage (Stage 5) will use `mp_bandgap_stage4_cleaned.csv` to build the composition- and structure-derived descriptors defined in the Stage 2 plan.

**Do not run Stage 5 code in this notebook.** Await confirmation before proceeding.
